导入标准模块:


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None
    display = None

STYLE_PATH = Path("../style/course.css")
TOGGLE_PATH = Path("../style/code_toggle.html")

if HTML is not None and display is not None:
    if STYLE_PATH.exists():
        display(HTML(f"<style>{STYLE_PATH.read_text(encoding='utf-8')}</style>"))
    if TOGGLE_PATH.exists():
        display(HTML(TOGGLE_PATH.read_text(encoding="utf-8")))

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True


本节建立在 [7.1 Jones 记号](7_1_jones_notation.ipynb)、[7.2 RIME](7_2_rime.ipynb) 和 [7.5 主波束](7_5_primary_beam.ipynb) 的基础上。那里我们已经介绍了 Jones 向量、Stokes 参数和方向依赖响应；这里则把焦点收缩到一个更具体、也更贴近观测的问题：

- 双极化馈源到底测到什么；
- 线极化馈源和圆极化馈源如何把 Stokes 信息编码到相关积里；
- 为什么视差角会让馈源坐标系中的极化信号随时间旋转；
- 为什么 D-term 泄漏会把 Stokes $I$ 污染进交叉手（cross-hand）和极化结果中。

极化问题里最容易让人迷失的一点，是**约定（convention）**非常多。本节只采用一套内部自洽的记号，并把重点放在“结构关系”和“可运行实验”上，而不是陷入不同学科之间的符号战争。


***


## 7.6 极化与天线馈源（C-Jones 与 D-Jones）

若天空方向上的亮度矩阵为 $\mathbf{B}$，天线 $p$ 的坐标/馈源变换记为 $\mathbf{C}_p$，馈源间泄漏记为 $\mathbf{D}_p$，则最简化的双极化测量方程可以写成

$$
\mathbf{V}_{pq}^{\rm obs} = \mathbf{D}_p \mathbf{C}_p \mathbf{B} \mathbf{C}_q^{H} \mathbf{D}_q^{H}.
$$

这里：

- `C-Jones` 负责描述坐标系、馈源基底和视差角之类的变换；
- `D-Jones` 负责描述两个正交馈源之间并不理想的相互泄漏；
- 最终相关器看到的四个相关积（如 `XX, XY, YX, YY` 或 `RR, RL, LR, LL`）都只是这个矩阵方程的不同分量。


In [ ]:
C_LINEAR_TO_CIRCULAR = (1.0 / np.sqrt(2.0)) * np.array(
    [[1.0, 1.0j], [1.0, -1.0j]], dtype=complex
)


def stokes_to_brightness_linear(I, Q, U, V):
    return 0.5 * np.array(
        [[I + Q, U + 1.0j * V], [U - 1.0j * V, I - Q]], dtype=complex
    )


def brightness_to_stokes_linear(B):
    XX, XY = B[0, 0], B[0, 1]
    YX, YY = B[1, 0], B[1, 1]
    I = np.real(XX + YY)
    Q = np.real(XX - YY)
    U = np.real(XY + YX)
    V = np.real(-1.0j * (XY - YX))
    return np.array([I, Q, U, V], dtype=float)


def linear_to_circular_brightness(B_linear):
    return C_LINEAR_TO_CIRCULAR @ B_linear @ C_LINEAR_TO_CIRCULAR.conj().T


def brightness_to_stokes_circular(B):
    RR, RL = B[0, 0], B[0, 1]
    LR, LL = B[1, 0], B[1, 1]
    I = np.real(RR + LL)
    Q = np.real(RL + LR)
    U = np.real(-1.0j * (RL - LR))
    V = np.real(RR - LL)
    return np.array([I, Q, U, V], dtype=float)


def feed_rotation_jones(chi_rad):
    return np.array(
        [
            [np.cos(chi_rad), np.sin(chi_rad)],
            [-np.sin(chi_rad), np.cos(chi_rad)],
        ],
        dtype=complex,
    )


def d_jones(dx, dy):
    return np.array([[1.0, dx], [dy, 1.0]], dtype=complex)


def observe_visibility(B_sky, Jp, Jq):
    return Jp @ B_sky @ Jq.conj().T


def stokes_from_linear_fraction(I, p_linear, evpa_deg, V=0.0):
    angle = np.deg2rad(evpa_deg)
    Q = I * p_linear * np.cos(2.0 * angle)
    U = I * p_linear * np.sin(2.0 * angle)
    return np.array([I, Q, U, V], dtype=float)


def matrix_product_vector(B):
    return np.array([B[0, 0], B[0, 1], B[1, 0], B[1, 1]], dtype=complex)


def annotate_heatmap(ax, values, fmt="{:.2f}"):
    for (row, col), value in np.ndenumerate(values):
        ax.text(col, row, fmt.format(value), ha="center", va="center", color="white")


### 7.6.1 极化基底与 C-Jones：线极化馈源和圆极化馈源

对双极化接收机来说，最常见的两种馈源基底是线极化基底 `(X, Y)` 和圆极化基底 `(R, L)`。它们之间的关系可以写成一个基底变换矩阵：

$$
\begin{pmatrix} E_R \\ E_L \end{pmatrix}
=
\mathbf{C}_{\rm lin\rightarrow circ}
\begin{pmatrix} E_X \\ E_Y \end{pmatrix},
\qquad
\mathbf{C}_{\rm lin\rightarrow circ}
=
\frac{1}{\sqrt{2}}
\begin{pmatrix}
1 & i \\
1 & -i
\end{pmatrix}.
$$

这正是 `C-Jones` 的一个典型用途：它并不代表“增益大小变化”，而是代表**坐标与基底的变化**。同一个天空源，在不同馈源基底里会有不同的相关积表示，但只要变换一致，恢复出来的 Stokes 参数应该完全相同。


In [ ]:
stokes_demo = np.array([1.0, 0.2, 0.3, 0.1])

B_linear = stokes_to_brightness_linear(*stokes_demo)
B_circular = linear_to_circular_brightness(B_linear)

stokes_back_from_linear = brightness_to_stokes_linear(B_linear)
stokes_back_from_circular = brightness_to_stokes_circular(B_circular)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

real_linear = np.real(B_linear)
imag_linear = np.imag(B_linear)
real_circular = np.real(B_circular)
imag_circular = np.imag(B_circular)

im0 = axes[0, 0].imshow(real_linear, cmap="coolwarm")
axes[0, 0].set_title("Linear-basis coherency: real part")
axes[0, 0].set_xticks([0, 1], labels=["X", "Y"])
axes[0, 0].set_yticks([0, 1], labels=["X", "Y"])
annotate_heatmap(axes[0, 0], real_linear)

im1 = axes[0, 1].imshow(imag_linear, cmap="coolwarm")
axes[0, 1].set_title("Linear-basis coherency: imag part")
axes[0, 1].set_xticks([0, 1], labels=["X", "Y"])
axes[0, 1].set_yticks([0, 1], labels=["X", "Y"])
annotate_heatmap(axes[0, 1], imag_linear)

im2 = axes[1, 0].imshow(real_circular, cmap="coolwarm")
axes[1, 0].set_title("Circular-basis coherency: real part")
axes[1, 0].set_xticks([0, 1], labels=["R", "L"])
axes[1, 0].set_yticks([0, 1], labels=["R", "L"])
annotate_heatmap(axes[1, 0], real_circular)

im3 = axes[1, 1].imshow(imag_circular, cmap="coolwarm")
axes[1, 1].set_title("Circular-basis coherency: imag part")
axes[1, 1].set_xticks([0, 1], labels=["R", "L"])
axes[1, 1].set_yticks([0, 1], labels=["R", "L"])
annotate_heatmap(axes[1, 1], imag_circular)

fig.tight_layout()
plt.show()
plt.close(fig)

print("原始 Stokes           =", np.array2string(stokes_demo, precision=3))
print("由线馈源矩阵恢复 Stokes =", np.array2string(stokes_back_from_linear, precision=3))
print("由圆馈源矩阵恢复 Stokes =", np.array2string(stokes_back_from_circular, precision=3))
print("两种基底恢复结果最大差值 =", np.max(np.abs(stokes_back_from_linear - stokes_back_from_circular)))


这一步的物理含义非常重要：**馈源基底改变了“信息分布方式”，但不改变天空本身。** 也就是说，线馈源和圆馈源的区别首先体现在“哪个相关积更直接地携带哪一种 Stokes 信息”，而不是“谁看到的是不同的宇宙”。


### 7.6.2 四个相关积如何编码 Stokes 信息

在线馈源基底中，

$$
\mathbf{B}_{\rm lin}
=
\frac{1}{2}
\begin{pmatrix}
I + Q & U + iV \\
U - iV & I - Q
\end{pmatrix},
$$

因而 `XX, YY` 主要携带 `I` 与 `Q`，而 `XY, YX` 主要携带 `U` 与 `V`。

在圆馈源基底中，

$$
\mathbf{B}_{\rm circ}
=
\frac{1}{2}
\begin{pmatrix}
I + V & Q + iU \\
Q - iU & I - V
\end{pmatrix},
$$

因而 `RR, LL` 主要携带 `I` 与 `V`，而 `RL, LR` 主要携带 `Q` 与 `U`。下面用几个理想化的纯态例子把这件事“点亮”。


In [ ]:
canonical_states = {
    "Unpolarized": np.array([1.0, 0.0, 0.0, 0.0]),
    "Pure Q": np.array([1.0, 0.6, 0.0, 0.0]),
    "Pure U": np.array([1.0, 0.0, 0.6, 0.0]),
    "Pure V": np.array([1.0, 0.0, 0.0, 0.4]),
}

linear_amplitudes = []
circular_amplitudes = []

for stokes in canonical_states.values():
    B_lin = stokes_to_brightness_linear(*stokes)
    B_circ = linear_to_circular_brightness(B_lin)
    linear_amplitudes.append(np.abs(matrix_product_vector(B_lin)))
    circular_amplitudes.append(np.abs(matrix_product_vector(B_circ)))

linear_amplitudes = np.array(linear_amplitudes).T
circular_amplitudes = np.array(circular_amplitudes).T

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

im0 = axes[0].imshow(linear_amplitudes, cmap="viridis", aspect="auto")
axes[0].set_title("Linear feeds: |XX, XY, YX, YY|")
axes[0].set_xticks(range(len(canonical_states)), labels=list(canonical_states.keys()), rotation=20)
axes[0].set_yticks(range(4), labels=["XX", "XY", "YX", "YY"])
annotate_heatmap(axes[0], linear_amplitudes)

im1 = axes[1].imshow(circular_amplitudes, cmap="viridis", aspect="auto")
axes[1].set_title("Circular feeds: |RR, RL, LR, LL|")
axes[1].set_xticks(range(len(canonical_states)), labels=list(canonical_states.keys()), rotation=20)
axes[1].set_yticks(range(4), labels=["RR", "RL", "LR", "LL"])
annotate_heatmap(axes[1], circular_amplitudes)

fig.tight_layout()
plt.show()
plt.close(fig)

print("各纯态在不同馈源基底中的相关积：")
for name, stokes in canonical_states.items():
    B_lin = stokes_to_brightness_linear(*stokes)
    B_circ = linear_to_circular_brightness(B_lin)
    print(f"\n{name}:")
    print("  linear  ->", np.array2string(matrix_product_vector(B_lin), precision=3))
    print("  circular->", np.array2string(matrix_product_vector(B_circ), precision=3))


这个图正好给出一个工程层面的直觉：

- 若你的科学目标非常关心 `Stokes V`，圆馈源会让它出现在平行手中；
- 若你更关心 `Q/U` 的恢复方式，圆馈源会把线偏振信息集中到交叉手；
- 线馈源和圆馈源并不存在绝对优劣，但它们确实会改变标定难点和误差传播路径。

也正因如此，阅读不同阵列文档时一定要先问清楚：它们的馈源基底是什么、相关器输出按什么顺序存放、以及所采用的 Stokes 约定到底是哪一套。


### 7.6.3 视差角与馈源坐标系旋转：另一个 C-Jones

对地平式天线而言，即使馈源本身始终固定在天线结构上，天空坐标系也会随跟踪相对馈源基底旋转。这会把天空中的 `Q/U` 混合成馈源坐标系中的 `Q'/U'`：

$$
\begin{pmatrix}
Q' \\ U'
\end{pmatrix}
=
\begin{pmatrix}
\cos 2\chi & \sin 2\chi \\
-\sin 2\chi & \cos 2\chi
\end{pmatrix}
\begin{pmatrix}
Q \\ U
\end{pmatrix},
$$

其中 $\chi$ 是视差角。它本质上依然是一个坐标变换，因此也可看成 `C-Jones` 的一部分。下面用一个固定天空源，直接展示馈源坐标系中测得的 `Q/U` 如何随旋转角变化。


In [ ]:
sky_stokes_rotation = stokes_from_linear_fraction(I=1.0, p_linear=0.12, evpa_deg=25.0, V=0.0)
B_sky_rotation = stokes_to_brightness_linear(*sky_stokes_rotation)

chi_deg = np.linspace(-90.0, 90.0, 181)
stokes_in_feed_frame = []
products_in_feed_frame = []

for chi in np.deg2rad(chi_deg):
    C_rot = feed_rotation_jones(chi)
    B_feed = C_rot @ B_sky_rotation @ C_rot.conj().T
    stokes_in_feed_frame.append(brightness_to_stokes_linear(B_feed))
    products_in_feed_frame.append(matrix_product_vector(B_feed))

stokes_in_feed_frame = np.array(stokes_in_feed_frame)
products_in_feed_frame = np.array(products_in_feed_frame)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(chi_deg, stokes_in_feed_frame[:, 1], label="Q in feed frame", lw=2.0)
axes[0].plot(chi_deg, stokes_in_feed_frame[:, 2], label="U in feed frame", lw=2.0)
axes[0].set_ylabel("Stokes value")
axes[0].set_title("Q/U mixing under feed rotation")
axes[0].legend()

axes[1].plot(chi_deg, np.real(products_in_feed_frame[:, 0]), label="Re(XX)", lw=2.0)
axes[1].plot(chi_deg, np.real(products_in_feed_frame[:, 3]), label="Re(YY)", lw=2.0)
axes[1].plot(chi_deg, np.real(products_in_feed_frame[:, 1]), label="Re(XY)", lw=1.8)
axes[1].set_xlabel("feed rotation angle chi [deg]")
axes[1].set_ylabel("correlation product")
axes[1].set_title("What the correlator sees in the feed basis")
axes[1].legend()

fig.tight_layout()
plt.show()
plt.close(fig)

print("天空本征 Stokes =", np.array2string(sky_stokes_rotation, precision=4))
print(
    f"馈源坐标系中 Q 的变化范围 = [{stokes_in_feed_frame[:, 1].min():.4f}, {stokes_in_feed_frame[:, 1].max():.4f}]"
)
print(
    f"馈源坐标系中 U 的变化范围 = [{stokes_in_feed_frame[:, 2].min():.4f}, {stokes_in_feed_frame[:, 2].max():.4f}]"
)


这解释了一个在极化标定里非常常见、但初学者经常困惑的现象：**同一个天空源的极化信号为什么在观测过程中看起来会变。** 答案并不是源自己在快速变化，而是馈源坐标系相对于天空在转。

也正因为 `Q/U` 会随 $2\chi$ 周期旋转，良好的视差角覆盖往往有助于把真实天空偏振与仪器泄漏分离开来。


### 7.6.4 D-Jones：馈源间泄漏如何把 Stokes I 污染成假极化

理想情况下，`X` 馈源只响应 `X` 分量，`Y` 馈源只响应 `Y` 分量。但真实系统中总会存在少量串扰，因此可以用

$$
\mathbf{D}
=
\begin{pmatrix}
1 & d_x \\
d_y & 1
\end{pmatrix}
$$

来表示泄漏。这里的 $d_x, d_y$ 通常是随频率缓慢变化的复数。它们会把本应只出现在平行手中的强 `Stokes I` 信号泄漏到交叉手里，从而伪造出极化。


In [ ]:
B_unpolarized = stokes_to_brightness_linear(1.0, 0.0, 0.0, 0.0)

Jp_leak = d_jones(0.03 + 0.02j, -0.015 + 0.01j)
Jq_leak = d_jones(-0.02 + 0.015j, 0.025 - 0.005j)

V_ideal = observe_visibility(B_unpolarized, np.eye(2, dtype=complex), np.eye(2, dtype=complex))
V_leaked = observe_visibility(B_unpolarized, Jp_leak, Jq_leak)
V_corrected = np.linalg.inv(Jp_leak) @ V_leaked @ np.linalg.inv(Jq_leak).conj().T

stokes_ideal = brightness_to_stokes_linear(V_ideal)
stokes_leaked = brightness_to_stokes_linear(V_leaked)
stokes_corrected = brightness_to_stokes_linear(V_corrected)

fig, ax = plt.subplots(figsize=(9, 4))
labels = ["I", "Q", "U", "V"]
x = np.arange(len(labels))
width = 0.25

ax.bar(x - width, stokes_ideal, width=width, label="ideal")
ax.bar(x, stokes_leaked, width=width, label="with D-terms")
ax.bar(x + width, stokes_corrected, width=width, label="after correction")
ax.set_xticks(x, labels=labels)
ax.set_ylabel("recovered Stokes")
ax.set_title("Leakage turns an unpolarized source into false polarization")
ax.legend()

fig.tight_layout()
plt.show()
plt.close(fig)

print("天线 p 的 D-term =", Jp_leak)
print("天线 q 的 D-term =", Jq_leak)
print("\n理想无偏振源恢复为      =", np.array2string(stokes_ideal, precision=5))
print("带 D-term 时表观 Stokes =", np.array2string(stokes_leaked, precision=5))
print("校正后恢复为           =", np.array2string(stokes_corrected, precision=5))


上面的结果很值得记住：即使天空源本来完全不偏振，几个百分点量级的 D-term 也足以制造出明显的假 `U/V`。这就是为什么极化标定从来不是“有空再做”的可选项，而是极化科学能否成立的前提。

反过来说，只要 D-term 模型足够准确，矩阵逆变换就能把这些仪器混合项再拆回去。


### 7.6.5 综合示例：不做校准时，Stokes 会随旋转角“漂移”

最后把前面的两个因素合在一起：设天空源有固定的本征极化，观测系统既存在馈源旋转，也存在 D-term 泄漏。若我们直接把相关器输出当成“已经是天空 Stokes”，结果会怎样？若我们先做 D-term 校正，再把馈源坐标系旋回天空坐标系，又会怎样？


In [ ]:
sky_stokes_true = np.array([1.0, 0.08, -0.06, 0.02])
B_sky_true = stokes_to_brightness_linear(*sky_stokes_true)

chi_demo_deg = np.linspace(-80.0, 80.0, 121)
raw_estimates = []
corrected_estimates = []

for chi in np.deg2rad(chi_demo_deg):
    C_rot = feed_rotation_jones(chi)
    Jp_total = Jp_leak @ C_rot
    Jq_total = Jq_leak @ C_rot

    V_obs = observe_visibility(B_sky_true, Jp_total, Jq_total)
    raw_estimates.append(brightness_to_stokes_linear(V_obs))

    V_cal = (
        np.linalg.inv(C_rot)
        @ np.linalg.inv(Jp_leak)
        @ V_obs
        @ np.linalg.inv(Jq_leak).conj().T
        @ np.linalg.inv(C_rot).conj().T
    )
    corrected_estimates.append(brightness_to_stokes_linear(V_cal))

raw_estimates = np.array(raw_estimates)
corrected_estimates = np.array(corrected_estimates)

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

for ax, index, label in zip(axes, [1, 2, 3], ["Q", "U", "V"]):
    ax.plot(chi_demo_deg, raw_estimates[:, index], label=f"raw {label}", lw=1.8, color="tab:red")
    ax.plot(chi_demo_deg, corrected_estimates[:, index], label=f"corrected {label}", lw=2.0, color="tab:blue")
    ax.axhline(sky_stokes_true[index], color="black", ls="--", lw=1.2, label=f"true {label}")
    ax.set_ylabel(label)
    ax.legend(loc="best")

axes[0].set_title("Calibration turns feed-frame measurements back into sky Stokes")
axes[-1].set_xlabel("feed rotation angle chi [deg]")

fig.tight_layout()
plt.show()
plt.close(fig)

print("天空真实 Stokes =", np.array2string(sky_stokes_true, precision=4))
print("\n不校准时的范围：")
for index, label in zip([1, 2, 3], ["Q", "U", "V"]):
    print(f"  {label}: [{raw_estimates[:, index].min():.4f}, {raw_estimates[:, index].max():.4f}]")

print("\n校准后的平均值与标准差：")
corrected_mean = corrected_estimates.mean(axis=0)
corrected_std = corrected_estimates.std(axis=0)
for index, label in enumerate(["I", "Q", "U", "V"]):
    print(f"  {label}: mean = {corrected_mean[index]:.5f}, std = {corrected_std[index]:.3e}")


这一节最后想让学生真正形成的判断是：

- 极化并不是四个 Stokes 符号的静态定义，而是一套**通过馈源基底和相关积编码进数据流**的测量过程；
- `C-Jones` 负责把天空坐标、馈源坐标和不同极化基底联系起来；
- `D-Jones` 负责描述不理想馈源导致的串扰，它会把强 `I` 泄漏成假极化；
- 若不做极化标定，相关器输出中的 `XX/XY/YX/YY` 绝不能直接等同于天空真实 Stokes；
- 真正可靠的极化成像，需要把基底变换、视差角旋转、D-term 和后续成像链路一起纳入统一的 RIME 思维框架。

到这里，第 7 章关于“仪器系统”的主线已经越来越完整：从 Jones 语言、RIME、模拟链路、数字相关器、主波束，再到馈源与极化，学生应当能够把“天空电磁场如何一路变成测量可见度”这件事看成一条连续而严谨的系统工程链条。
